# ECG → LVEF category: train with ImageNet-pretrained CNNs on a free Colab GPU

**Before running:** menu **Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**.

This notebook:
1. downloads the project (code + ECG PDFs) from GitHub,
2. builds the filtered ECG images, using the **same patient-grouped train/val/test split** as the results in the README,
3. trains image CNNs **starting from ImageNet weights** (`--pretrained`) on the GPU,
4. re-fits the header model and fuses it with the best pretrained CNN,
5. prints the updated comparison table and lets you download the results.

Rough time estimate on a T4 (not measured): 30–60 minutes in total.

In [ ]:
# 1. Check that a GPU is attached (should print e.g. 'GPU 0: Tesla T4')
!nvidia-smi -L

In [ ]:
# 2. Get the code and data (public repo, branch with this project)
!git clone --depth 1 -b claude/funny-gauss-udlmv0 https://github.com/krithikareddy22/ECG-Image-Based-Prediction-EF-hear-attack-12-Lead.git ecg
%cd ecg
!pip install -q timm pymupdf scikit-image tabulate

In [ ]:
# 3. Render + filter the ECG images (reuses results/split.csv, so the test set is identical)
!python -m ecg_ef.preprocess

## 4. Train pretrained image CNNs
Each model starts from ImageNet weights (downloaded automatically from Hugging Face) and is fine-tuned with lr 3e-4.
Results go to `results/<model>_filtered_pretrained/`.

In [ ]:
import torch; print('CUDA available:', torch.cuda.is_available())
for m in ['resnet18', 'resnet34', 'efficientnet_b0', 'convnext_atto', 'densenet121']:
    !python -m ecg_ef.train --model {m} --pretrained

## 5. Header model + fusion with the best pretrained CNN (chosen on the validation set)

In [ ]:
!python -m ecg_ef.header
import json, pathlib
from ecg_ef.train import selection_score
tags = [p.parent.name for p in pathlib.Path('results').glob('*_pretrained/metrics.json')]
val = {t: selection_score(json.load(open(f'results/{t}/metrics.json'))['val']) for t in tags}
best = max(val, key=val.get)
print('validation scores:', {k: round(v, 3) for k, v in val.items()})
print('best pretrained CNN:', best)
!python -m ecg_ef.ensemble --members header_logreg {best} --tune-weight --name fusion_header_pretrained

## 6. Updated comparison (all models, same 210 test ECGs)

In [ ]:
!python -m ecg_ef.compare > compare.log
from IPython.display import Image, Markdown, display
display(Image('results/comparison.png'))
import pandas as pd
cols = ['Model', 'Input', 'Val score', 'Accuracy', 'Balanced acc', 'Macro-F1', 'HF AUC', 'HF sensitivity', 'HF specificity']
display(pd.read_csv('results/comparison.csv')[cols].round(3))
print(open('results/best_model.json').read())

## 7. Download the results
Downloads a zip with `results/` and the trained models. Keep it: Colab deletes everything when the session ends.

In [ ]:
!zip -qr ecg_results.zip results models
from google.colab import files
files.download('ecg_results.zip')

## 8. (Optional) Predict your own ECG PDFs with the selected model

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose one or more ECG PDF / PNG / JPG files
for name in uploaded:
    !python -m ecg_ef.predict "{name}"